一个完整的Agent至少要包含两个关键的部分：
- 模型：是Agent的大脑，负责推理、分析，规划任务步骤
- 工具：是Agent的手脚，负责执行任务，与外界交互

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# 1.使用tool装饰器定义工具
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"

In [5]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 2.创建智能体，并绑定工具
agent = create_agent(
    model="deepseek-chat",
    tools=[get_weather]
)

# 3.调用Agent
response = agent.invoke(
    {"messages": [HumanMessage(content="杭州今天天气如何?")]},
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

杭州今天天气如何?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_fXJXnWlt3o81UHsi28Gi5985)
 Call ID: call_00_fXJXnWlt3o81UHsi28Gi5985
  Args:
    location: 杭州
================================= Tool Message =================================
Name: get_weather

Current weather in 杭州 is sunny
================================== Ai Message ==================================

杭州今天天气晴朗（sunny）。☀️

如果需要了解气温、湿度或未来几天的预报，可以告诉我，我再帮你查询。


所谓的工具，本质就是一个可调用的函数，要想让Agent知道有哪些工具可调用，该如何调用这些工具，就必须把这个函数的详细信息发送给模型。包括：
- 函数名
- 函数的作用
- 函数的参数和返回值信息

In [7]:
# 定义工具
from langchain.tools import tool


#1.1基于tool装饰器定义工具
@tool("square_root", description="Calculate the square root of a number")
def square_root(x: float) -> float:
    return x ** 0.5
#1.2手使用函数名和文档注释描述工具
@tool
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5


如果要覆盖工具的入参信息则会复杂很多，我们要借助于Pydantic或JSON约束。

例如，我们需要定义个查询天气的tool，借助于Pydantic来约束入参。
我们定义一个入参的模型，在模型中添加入参描述信息：

In [ ]:
from langchain.tools import tool
from pydantic import BaseModel, Field
from typing import Literal
# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )
# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

In [8]:
# 调用数学工具
tool1.invoke({"x": 467})

21.61018278497431

In [9]:
# 调用查询天气工具
get_weather.invoke({"location": "杭州", "include_forecast": True})

'Current weather in 杭州 is sunny'

In [10]:
from langchain.agents import create_agent

# 创建智能体，并添加工具
agent = create_agent(
    model="deepseek-chat",
    tools=[tool1, get_weather],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

In [14]:
# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="467和122的平方根是多少?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

21.6101827849743111.045361017187261两个数的平方根分别是：

- **√467 ≈ 21.6102**
- **√122 ≈ 11.0454**

In [15]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="北京和杭州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

Current weather in 北京 is sunnyCurrent weather in 杭州 is sunny根据查询结果：

- **北京**：当前天气晴朗 ☀️
- **杭州**：当前天气晴朗 ☀️

不过我的天气工具只能提供**当前实时天气**，无法获取未来几天的预报。如果你需要未来几天的详细天气趋势（气温、降水概率等），建议查看中国天气网或手机天气 App 获取逐日预报。

需要我帮你查其他城市的实时天气吗？